In [2]:
# ============================================
# 1. INSTALL REQUIRED PACKAGES
# ============================================

!pip install -q groq "mcp[cli]>=1.28,<2"

# ============================================
# 2. IMPORT LIBRARIES
# ============================================

import os
import json
import time
import sys
import asyncio
import subprocess
from getpass import getpass
from groq import Groq
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

# ============================================
# 3. GROQ API KEY
# ============================================

api_key = getpass("Enter your Groq API Key: ")

os.environ["GROQ_API_KEY"] = api_key

# ============================================
# 4. CREATE MCP SERVER FILE
# ============================================

server_code = '''
import os
import json
from groq import Groq
from mcp.server.fastmcp import FastMCP

client = Groq(api_key=os.environ["GROQ_API_KEY"])

MODEL = "openai/gpt-oss-120b"

mcp = FastMCP("Nutrition AI Agent")


@mcp.tool()
def analyze_food(food: str) -> str:
    """
    Analyze a food item and return exactly 6 important
    nutritional details.
    """

    prompt = f"""
    You are a professional nutrition analysis assistant.

    Analyze the following food: {food}

    Assume a serving size of 100 grams unless the food
    is normally measured differently.

    Return valid JSON with exactly these 6 fields:

    {{
        "food": "Food name and serving size",
        "calories": "Calories in kcal",
        "protein": "Protein in grams",
        "carbohydrates": "Carbohydrates in grams",
        "fat": "Fat in grams",
        "vitamins_minerals": "Important vitamins and minerals",
        "health_benefits": "Health benefits"
    }}

    Rules:
    - Always use the same 7 JSON fields shown above.
    - Give approximate nutritional values.
    - Specify whether the food is raw or cooked when relevant.
    - Do not invent exact values when reliable data is unavailable.
    - Keep the health benefits concise.
    - Return JSON only.
    """

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ],
        response_format={"type": "json_object"},
        temperature=0.2
    )

    data = json.loads(response.choices[0].message.content)

    output = f"""
    NUTRITION ANALYSIS

    Food: {data.get('food', food)}

    1. Calories:
       {data.get('calories', 'Not available')}

    2. Protein:
       {data.get('protein', 'Not available')}

    3. Carbohydrates:
       {data.get('carbohydrates', 'Not available')}

    4. Fat:
       {data.get('fat', 'Not available')}

    5. Vitamins & Minerals:
       {data.get('vitamins_minerals', 'Not available')}

    6. Health Benefits:
       {data.get('health_benefits', 'Not available')}
    """

    return output


if __name__ == "__main__":
    mcp.run(transport="streamable-http")
'''

with open("nutrition_server.py", "w") as file:
    file.write(server_code)

print("MCP Server file created successfully!")


# ============================================
# 5. START MCP SERVER
# ============================================

# Stop an older server started in this notebook, if any.
try:
    if server_process.poll() is None:
        server_process.terminate()
        server_process.wait(timeout=5)
except (NameError, Exception):
    pass

server_process = subprocess.Popen(
    [sys.executable, "nutrition_server.py"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env=os.environ.copy()
)

time.sleep(5)

if server_process.poll() is not None:
    raise RuntimeError("MCP Server failed to start. Check the server configuration.")

print("MCP Server started successfully!")


# ============================================
# 6. CONNECT MCP CLIENT
# ============================================

async def analyze_with_mcp(food):

    async with streamable_http_client(
    "http://127.0.0.1:8000/mcp"
        ) as (read, write, _):
        async with ClientSession(read, write) as session:

            await session.initialize()

            result = await session.call_tool(
                "analyze_food",
                arguments={
                    "food": food
                }
            )

            if result.isError:
                print("Error:", result.content)
                return

            for content in result.content:
                if content.type == "text":
                    print(content.text)


# ============================================
# 7. GET USER INPUT
# ============================================

food_name = input("Enter a food name: ")

if food_name.strip():
    await analyze_with_mcp(food_name.strip())
else:
    print("Please enter a valid food name.")

Enter your Groq API Key: ··········
MCP Server file created successfully!
MCP Server started successfully!
Enter a food name: chicken

    NUTRITION ANALYSIS

    Food: Chicken, roasted (skinless), 100 g

    1. Calories:
       165 kcal

    2. Protein:
       31 g

    3. Carbohydrates:
       0 g

    4. Fat:
       3.6 g

    5. Vitamins & Minerals:
       Niacin (B3), Vitamin B6, Selenium, Phosphorus, Zinc

    6. Health Benefits:
       Provides high‑quality protein for muscle maintenance; supports metabolism and immune function; low in carbs
    


In [9]:
import mcp.client.streamable_http as http

print(dir(http))

['AsyncGenerator', 'Awaitable', 'CONNECTION_CLOSED', 'Callable', 'ClientMessageMetadata', 'ContextReceiveStream', 'ContextSendStream', 'DEFAULT_MAX_SSE_EVENT_SIZE', 'DEFAULT_RECONNECTION_DELAY_MS', 'ErrorData', 'EventSource', 'INTERNAL_ERROR', 'INVALID_REQUEST', 'JSONRPCError', 'JSONRPCMessage', 'JSONRPCNotification', 'JSONRPCRequest', 'JSONRPCResponse', 'LAST_EVENT_ID', 'MAX_RECONNECTION_ATTEMPTS', 'MCP_PROTOCOL_VERSION_HEADER', 'MCP_SESSION_ID', 'METHOD_NOT_FOUND', 'MODERN_PROTOCOL_VERSIONS', 'PARSE_ERROR', 'RequestContext', 'RequestId', 'ResumptionError', 'ServerSentEvent', 'SessionMessage', 'SessionMessageOrError', 'StreamReader', 'StreamWriter', 'StreamableHTTPError', 'StreamableHTTPTransport', 'TaskGroup', 'TransportStreams', 'ValidationError', '_InFlightPost', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '_annotations', '_unfollowed_redirect', 'anyio', 'asynccontextmanager', 'cancelled_request_id_from_params', 'context

In [10]:
import importlib.metadata

print(importlib.metadata.version("mcp"))

1.30.0


In [11]:
from mcp.client.streamable_http import streamable_http_client